<a href="https://colab.research.google.com/github/milicadzodan/gisp24/blob/main/docs/notebooks/00_ee_auth_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee
import geemap

In [2]:
ee.Authenticate()
ee.Initialize(project="ee-milicadzodan49")

In [3]:
import ee
import geemap

# Granica Srbije
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

serbia = countries.filter(
    ee.Filter.eq('country_na', 'Serbia')
).geometry()

In [4]:
era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

In [5]:
temp_1950_2000 = (
    era5
    .filterDate('1950-01-01', '2000-12-31')
    .select('temperature_2m')
    .mean()
    .subtract(273.15)      # Kelvin -> Celzijus
    .clip(serbia)
)

In [6]:
temp_2025 = (
    era5
    .filterDate('2025-01-01', '2025-12-31')
    .select('temperature_2m')
    .mean()
    .subtract(273.15)
    .clip(serbia)
)

In [7]:
difference = temp_2025.subtract(temp_1950_2000)

In [8]:
Map = geemap.Map(center=[44.0, 20.8], zoom=7)

tempVis = {
    'min': -5,
    'max': 25,
    'palette': [
        '0000ff',
        '00ffff',
        '00ff00',
        'ffff00',
        'ff9900',
        'ff0000'
    ]
}

diffVis = {
    'min': -3,
    'max': 3,
    'palette': [
        '0000ff',
        'ffffff',
        'ff0000'
    ]
}

Map.addLayer(temp_1950_2000, tempVis, '1950-2000')
Map.addLayer(temp_2025, tempVis, '2025')
Map.addLayer(difference, diffVis, 'Difference')

Map

Map(center=[44.0, 20.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [9]:
novi_sad = ee.Geometry.Point([19.8335, 45.2671])

In [10]:
value1 = temp_1950_2000.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=novi_sad,
    scale=1000
)

print("1950-2000:")
print(value1.getInfo())

1950-2000:
{'temperature_2m': 11.657328059780127}


In [11]:
value2 = temp_2025.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=novi_sad,
    scale=1000
)

print("2025:")
print(value2.getInfo())

2025:
{'temperature_2m': 13.410249836454227}


In [12]:
value3 = difference.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=novi_sad,
    scale=1000
)

print("Difference:")
print(value3.getInfo())

Difference:
{'temperature_2m': 1.7529217766740999}


In [13]:
t1 = value1.getInfo()['temperature_2m']
t2 = value2.getInfo()['temperature_2m']
td = value3.getInfo()['temperature_2m']

print(f"Prosek 1950-2000: {t1:.2f} °C")
print(f"Prosek 2025: {t2:.2f} °C")
print(f"Razlika: {td:.2f} °C")

Prosek 1950-2000: 11.66 °C
Prosek 2025: 13.41 °C
Razlika: 1.75 °C
